## Model EDA
- Input: UPDATEDOATH_cleaned_demographic_merge_training_nodates.V2.csv contains: 
  - Tax IDs from raw civil citations
  - Corresponding OATH records based on ticket numbers
  - Disciplinary Action data based on officer tax id 
  - Award data based on offcier tax id
  - Charge data based on officer tax id
  - Arrest data based on officer tax id
  - Allegations data based on officer tax id

Through this notebook, we are looking at the (1) validity of variables, (2) whether to keep or delete variables, (3) standardize/mutate  variables, and (4) add additional/most recent officer variables suitable for an analytical dataset. 

- Output: civil_summons_2021_v1.csv contains:
  - Cleaned OATH variables 
  - Most recent legal aid roster data based on officer tax id
  - Most recent legal aid payrool data based on offcier tax id/officer name 

https://github.com/the-legal-aid-society/LELU/blob/main/README.md 

### Original 2021 Civil Citations EDA

In [ ]:
# set up 
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import os
os.getcwd()

In [ ]:
os.chdir("/Users/joyamelvin/Desktop/Evidence for Justice Lab/civil-summons-project/")
os.getcwd()

In [ ]:
# Merging original NYPD data
# Load the datasets
oath = pd.read_csv('data/UPDATEDOATH_cleaned_demographic_merge_training_nodates.V2.csv', dtype={'Ticket_Number': str}, low_memory=False)
nypd = pd.read_csv('data/Search_NYPD Civil Summons Case Info (Jan 1 2021 to Aug 31 2022).xlsx - Sheet1.csv', dtype={'Ticket Number': str}, low_memory=False)
oath.Ticket_Number.describe() 

In [ ]:
# Checking dupes 
dupes = oath[oath['Ticket_Number'].duplicated(keep=False)].sort_values('Ticket_Number')
dupes
print(f"Duplicate rows: {len(oath) - oath['Ticket_Number'].nunique()}")

In [ ]:
# All fields consistent; drop duplicates
dupes.to_csv('data/duplicates_3.29.26.csv', index=False)
oath = oath.drop_duplicates(subset='Ticket_Number', keep='first')

In [ ]:
oath.Ticket_Number.describe() 

In [ ]:
# Merge civil citations to correct nypd oath file 
# Clean ticket numbers
oath['Ticket_Number'] = oath['Ticket_Number'].str.strip().str.lstrip('0')
nypd['Ticket Number'] = nypd['Ticket Number'].str.strip().str.lstrip('0')

# Rename all NYPD columns with O_ prefix, skip the merge key
nypd_renamed = nypd.rename(columns={
    col: 'O_' + col.replace(' ', '_').replace('(', '').replace(')', '').replace('#', 'Num').replace(':', '').replace('/', '_').replace(',', '')
    for col in nypd.columns if col != 'Ticket Number'
})
nypd_renamed = nypd_renamed.rename(columns={'Ticket Number': 'Ticket_Number'})

# Drop any duplicate columns in nypd_renamed before merging
nypd_renamed = nypd_renamed.loc[:, ~nypd_renamed.columns.duplicated()]

# Keep only non-O_ prefixed columns from OATH
oath_keep_cols = ['Ticket_Number'] + [col for col in oath.columns 
                                       if not col.startswith('O_') and not col.startswith('o_')]
oath_subset = oath[oath_keep_cols]

# Drop any duplicate columns in oath_subset before merging
oath_subset = oath_subset.loc[:, ~oath_subset.columns.duplicated()]

# Inner join
civil_summons_2021 = oath_subset.merge(nypd_renamed, on='Ticket_Number', how='inner')

print(f"OATH rows:            {len(oath)}")
print(f"NYPD rows:            {len(nypd)}")
print(f"Matched/merged rows:  {len(civil_summons_2021)}")
print(f"Dropped from OATH:    {len(oath) - len(civil_summons_2021)}")
print(f"Total columns:        {len(civil_summons_2021.columns)}")

civil_summons_2021.to_csv('data/civil_summons_2021_merged.csv', index=False)

In [ ]:
# Check September in the original OATH file before merge
sept_oath = oath[oath['O_Violation_Date'].str.startswith('9/')]
print(f"September citations in OATH (before merge): {len(sept_oath)}")

# 475 spetember citations in the original oath file 
sept_nypd = nypd[nypd['Violation Date'].str.startswith('9/')]
print(f"September citations in NYPD (before merge): {len(sept_nypd)}")

# Check September in merged file
sept_merged = civil_summons_2021[civil_summons_2021['O_Violation_Date'].str.startswith('9/')]
print(f"September citations in merged file: {len(sept_merged)}")

In [ ]:
# inspect september oath files 
sept_nypd[['Ticket Number']].to_csv('data/september_tickets.csv', index=False)

In [ ]:
# Missing dataframe 
pd.set_option('display.max_rows', None)

missing_df = pd.DataFrame({
    'missing_count': civil_summons_2021.isnull().sum(),
    'missing_pct': civil_summons_2021.isnull().mean().mul(100).round(2)
}).sort_values('missing_pct', ascending=True)

missing_df 

In [ ]:
# 2423 unique offers in our civil summons data; we want this number to be assigned to most variables.  
civil_summons_2021.Final_Officer_Tax_ID.nunique()

In [ ]:
# 4642 total civil citations matches - the same 
civil_summons_2021.Ticket_Number.count()

In [ ]:
civil_summons_2021.columns.tolist()

In [ ]:
civil_summons_2021.shape

In [ ]:
# 2423 unique offers in our civil summons data; we want this number to be assigned to most behavioral variables.  
civil_summons_2021.Final_Officer_Tax_ID.nunique()

In [ ]:
civil_summons_2021.Tax_Registry_Number_Tax_ID.describe()

In [ ]:
# Quick turn these columns from object into int64 

cols = [
    'Final_Officer_Tax_ID',
    'aa_felony_arrest_count',
    'aa_infraction_arrest_count',
    'aa_misdemeanor_arrest_count',
    'aa_other_arrest_count',
    'aa_violation_arrest_count'
]

for col in cols:
    civil_summons_2021[col] = (
        pd.to_numeric(civil_summons_2021[col], errors='coerce')
          .fillna(0)
          .astype('int64')
    )

In [ ]:
civil_summons_2021.master_command.head()

### 2021 OATH EDA 
The OATH Hearings Division Case Status dataset contains information about alleged public safety and quality of life violations that are filed and adjudicated through the City’s administrative law court, the NYC Office of Administrative Trials and Hearings (OATH) and provides information about the infraction charged, decision outcome, payments, amounts and fees relating to the each civil summons ticket. This data was merged via ticket_number.


#### O_Violation_Date : The date on which the alleged violation indicated on the summons occurred
ANALYSIS: Make categorical; More citations given during the colder months

In [ ]:
civil_summons_2021['O_Violation_Date'].head()

In [ ]:
civil_summons_2021['O_Violation_Date'].isna().sum()

In [ ]:
# Where are September Citations
import matplotlib.pyplot as plt 
import matplotlib.ticker as ticker

# Extract month and year separately from full date string (m/d/yyyy)
civil_summons_2021['month_year'] = civil_summons_2021['O_Violation_Date'].str.extract(r'(\d+)/\d+/(\d+)').apply(
    lambda x: pd.to_datetime(f"{x[0]}/{x[1]}", format='%m/%Y') if pd.notna(x[0]) else pd.NaT, axis=1
)

monthly_counts = civil_summons_2021['month_year'].value_counts().sort_index()

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

ax.bar(monthly_counts.index, monthly_counts.values, width=20, color='steelblue', edgecolor='white')

ax.set_title('Civil Summons Citations by Month', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Number of Citations', fontsize=12)

ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(plt.matplotlib.dates.MonthLocator())
plt.xticks(rotation=45, ha='right')

ax.yaxis.set_major_locator(ticker.MultipleLocator(50))
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Save September ticket numbers from NYPD
sept_nypd_tickets = set(nypd[nypd['Violation Date'].str.startswith('9/')]['Ticket Number'].str.strip().str.lstrip('0'))
print(f"September NYPD ticket numbers: {len(sept_nypd_tickets)}")

# Get all ticket numbers from OATH
oath_tickets = set(oath['Ticket_Number'].str.strip().str.lstrip('0'))
print(f"Total OATH ticket numbers: {len(oath_tickets)}")

# Find overlap
overlap = sept_nypd_tickets.intersection(oath_tickets)
print(f"September NYPD tickets that match OATH: {len(overlap)}")

# Show some examples of September NYPD tickets vs OATH tickets
print(f"\nSample September NYPD tickets: {list(sept_nypd_tickets)[:5]}")
print(f"Sample OATH tickets: {list(oath_tickets)[:5]}")

In [ ]:
oath.Ticket_Number.describe()

In [ ]:
# O_Violation_Date : The date on which the alleged violation indicated on the summons occurred
# IMPORTANT: More citations given during the colder months

civil_summons_2021['O_Violation_Date'] = pd.to_datetime(
    civil_summons_2021['O_Violation_Date'],
    format='%m/%d/%Y',  
    errors='coerce'
)


In [ ]:
# Making sure my dates are counted correctly; 
civil_summons_2021['O_Violation_Date'].head(10)
civil_summons_2021['O_Violation_Date'].isna().sum()

In [ ]:
# Distribution of dates 
civil_summons_2021['O_Violation_Date'].hist(figsize=(12,5), bins=50)
plt.title("Distribution of Citations Given by Date")
plt.xlabel("Date")
plt.ylabel("Number of Citations Given")

plt.savefig('EDA_figures/Citations_Given_by_Date.png', dpi=300, bbox_inches='tight')
plt.show()

#### O_Violation_Date --> month_issued 
- Creating numerical variable from violation date for each month ticket was issued
  

In [ ]:
civil_summons_2021['O_Violation_Date'].value_counts()

In [ ]:
civil_summons_2021['month_issued'] = pd.to_datetime(civil_summons_2021['O_Violation_Date']).dt.month

In [ ]:
civil_summons_2021['month_issued'].value_counts()

In [ ]:
# Find rows where month_issued is NaN or 0
missing_mask = civil_summons_2021['month_issued'].isna() | (civil_summons_2021['month_issued'] == 0.0)

# Count them
print(f"Count of NaN values: {civil_summons_2021['month_issued'].isna().sum()}")
print(f"Count of 0 values: {(civil_summons_2021['month_issued'] == 0.0).sum()}")
print(f"Total missing/zero: {missing_mask.sum()}")

# Show the corresponding O_Violation_Date values and Ticket_Number
civil_summons_2021[missing_mask][['O_Violation_Date', 'month_issued', 'Ticket_Number', ]]

#### O_Violation_Time : The time of day at which the alleged violation indicated on the summons occurred
ANALYSIS: More citations in the evening and the afternoon, in comparison to the morning, night, and early morning. 

In [ ]:
civil_summons_2021['O_Violation_Time'].head(10)

In [ ]:
# O_Violation_Time : The time of day at which the alleged violation indicated on the summons occurred
# IMPORTANT: 
civil_summons_2021['Violation_Hour'] = pd.to_datetime(
    civil_summons_2021['O_Violation_Time'],
    format="%H:%M:%S",
    errors="coerce"
).dt.hour

# Making sure my dates are counted correctly
civil_summons_2021['Violation_Hour'].head(10)
civil_summons_2021['Violation_Hour'].isna().sum()


In [ ]:
civil_summons_2021[civil_summons_2021['Violation_Hour'].isna()]


In [ ]:
# Binning time by early morning (00-05); Morning (06-11); Afternoon (12-16); Evening (12-20); Night (21-23) 

# Function for binning hours 
def time_of_day(hour):
    if pd.isna(hour):
        return None
    if 0 <= hour < 6:
        return "Early Morning"
    elif 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

civil_summons_2021['O_Violation_Time_Category'] = civil_summons_2021['Violation_Hour'].apply(time_of_day)

In [ ]:
civil_summons_2021['O_Violation_Time_Category'].value_counts()


In [ ]:
civil_summons_2021['O_Violation_Time_Category'].value_counts().plot(
    kind='bar',
    figsize=(8,4),
    title='Distribution of Summons by Time of Day'
)
plt.xlabel('Time of Day')
plt.ylabel('Count')

plt.savefig('EDA_figures/Distribution of Summons by Time of Day.png', dpi=300, bbox_inches='tight')
plt.show()


#### O_Balance_Due: The balance of the monetary penalty owed by the respondent
**DELETE**


ANALYSIS: Average balance due is around $16; matches literature that most citations result in no penalty fine, if the burden is resolved (which is the catch)

In [ ]:
# OATH Balance Due Distribution 
# How can you have a -500$ balance; could be a data collection issue here
civil_summons_2021['O_Balance_Due'].describe()

In [ ]:
# O Balance Due Distribution
civil_summons_2021['O_Balance_Due'] = pd.to_numeric(
    civil_summons_2021['O_Balance_Due'], errors='coerce'
)

# plot 
civil_summons_2021.O_Balance_Due.hist(
    bins=20, 
    edgecolor='black', 
    figsize=(8,5)
)
plt.title("Distribution of Monetary Penalty Owed by Respondent")
plt.xlabel("Outstanding Balance Due ($)")
plt.ylabel("Number of Summonses")
plt.savefig('EDA_figures/Distribution of Monetary Penalty Owed by Respondent.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
money_cols = ['O_Balance_Due', 'O_Total_Violation_Amount', 'O_Penalty_Imposed', 
              'O_Paid_Amount', 'O_Additional_Penalties_or_Late_Fees',
              'O_Charge_Num1_Infraction_Amount', 'O_Charge_Num2_Infraction_Amount', 'O_Charge_Num3_Infraction_Amount']

for col in money_cols:
    civil_summons_2021[col] = pd.to_numeric(civil_summons_2021[col], errors='coerce')

print(civil_summons_2021[money_cols].dtypes)

In [ ]:
# quick sanity check on the distributions
civil_summons_2021[money_cols].describe()

#### O_Violation_Location_Borough: One of the five boroughs of New York City (Brooklyn, Bronx, Manhattan, Queens and Staten Island) in which the alleged violation occurred
ANALYSIS: Most citations given in Manhattan, Brooklyn, Queens. Minimal citations given in Bronx, Not NYC, and Staten Island


In [ ]:
# Citation Borough Locations
civil_summons_2021.O_Violation_Location_Borough.hist(
    bins=50, 
    edgecolor='black', 
    figsize=(8,5)
)
plt.title("Distribution of Citation by Borough Location")
plt.xlabel("Borough Location")
plt.ylabel("Number of Summonses")


plt.savefig('EDA_figures/Distribution of Citation by Borough Location.png', dpi=300, bbox_inches='tight')
plt.show()

#### O_Violation_Location_City: Name of the City, town or community where the alleged violation occurred

**DELETE**


ANALYSIS: Too skewed for analysis; 360 missing values, has a mixture of boroughs in addition to neighborhoods. Unreliable due to location based on officer knowledge of area. 

In [ ]:
civil_summons_2021['O_Violation_Location_City'].value_counts()

#### O_Violation_Location_Zip_Code: The zip code of the address where the alleged violation occurred

**DELETE**

ANALYSIS: Unreliable, 3,067 missing values

In [ ]:
civil_summons_2021['O_Violation_Location_Zip_Code'].value_counts()

#### O_Respondent_Address_Borough: The borough in which the respondent lives or is located.
ANALYSIS: More people not from NYC receive civil summons, Queens, Brooklyn. Less residents from Bronx, Manhattan, and Staten Island received civil citations in 2021.

In [ ]:
# Significant change 
# Citation Borough Locations
civil_summons_2021.O_Respondent_Address_Borough.hist(
    bins=50, 
    edgecolor='black', 
    figsize=(8,5)
)
plt.title("Recipients by Borough")
plt.xlabel("Borough Location")
plt.ylabel("Number of Summonses")
plt.show()

#### O_Hearing_Status: Hearing status
- Hearing Completed: hearing was completed, violation resolved
- Paid in Full: Paid, violation resolved
- Defaulted: Did not appear/Unpaid; you can either pay the default penalty or complete a Request for a New Hearing After Failure to Appear form. If your Request is granted, you will be given a date to appear for a hearing to contest the summons.
- Docketed: Unpaid; the City may attempt to collect on the money owed through court proceedings or by attaching liens to real property
- New Issuance: No respondent action yet 
- Appeal Decision Rendered: Respondend tfiled an appeal 
- Assigned: Case has been assigned to a eharing officer/respondent has not yet completed a hearing 

ANALYSIS: 2430 of the hearings were complete; 964 were paid in full; 775 were defaulted; 385 were docketed; 103 were new issuances; 5 appealed: 1 assigned 

In [ ]:
civil_summons_2021['O_Hearing_Status'].value_counts()

In [ ]:
# O hearing Status Distribution
status_counts = civil_summons_2021['O_Hearing_Status'].value_counts().reset_index()
status_counts.columns = ['Hearing Status', 'Number of Summonses']

# Plot
plt.figure(figsize=(10,6))
sns.barplot(
    data=status_counts, 
    x='Hearing Status', 
    y='Number of Summonses', 
    color='steelblue', 
    edgecolor='black'
)

plt.title("Distribution of Civil Citation Hearing Statuses", fontsize=14, fontweight='bold')
plt.xlabel("Hearing Status", fontsize=12)
plt.ylabel("Number of Summonses", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

plt.savefig('EDA_figures/Distribution of Civil Citation Hearing Statuses.png', dpi=300, bbox_inches='tight')
plt.show()


#### O_Hearing_Result: Outcome or result of the hearing 

**DELETE**

ANALYSIS: 844 missing results; 2377 were dismissed; 1161 defaulted; 250 in violation; 27 settled in-vio; 4 adjourned 

In [ ]:
civil_summons_2021['O_Hearing_Result'].value_counts()

In [ ]:
# O_Hearing_Results 
status_counts = civil_summons_2021['O_Hearing_Result'].value_counts().reset_index()
status_counts.columns = ['Hearing Result', 'Number of Summonses']

# Plot
plt.figure(figsize=(10,6))
sns.barplot(
    data=status_counts, 
    x='Hearing Result', 
    y='Number of Summonses', 
    color='steelblue', 
    edgecolor='black'
)

plt.title("Distribution of Civil Citation Hearing Results", fontsize=14, fontweight='bold')
plt.xlabel("Hearing Results", fontsize=12)
plt.ylabel("Number of Summonses", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('EDA_figures/Distribution of Civil Citation Hearing Results.png', dpi=300, bbox_inches='tight')
plt.show()


#### O_Total_Violation_Amount: The monetary amount associated with the violation
ANALYSIS: This is total violation amount charged; most are around 250, 0, 25, 100. 

In [ ]:
civil_summons_2021['O_Total_Violation_Amount'].value_counts()

In [ ]:
data = civil_summons_2021['O_Total_Violation_Amount'].value_counts().reset_index()
data.columns = ['Hearing Result', 'Number of Summonses']

# Plot
plt.figure(figsize=(10,6))
sns.barplot(
    data=status_counts, 
    x='Hearing Result', 
    y='Number of Summonses', 
    color='steelblue', 
    edgecolor='black'
)

plt.title("Distribution of Charged Amount for Each Citation", fontsize=14, fontweight='bold')
plt.xlabel("Amount", fontsize=12)
plt.ylabel("Number of Summonses", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('EDA_figures/Distribution of Charged Amount for Each Citation.png', dpi=300, bbox_inches='tight')
plt.show()

#### O_Payment_Status <- O_Penalty_Imposed: The actual penalty the respondent is required to pay & O_Paid_Amount: The total amount paid by the respondent

ANALYSIS: Different from hearing status, directly indicates willingness to pay. Created a new variable "Payment-Status" which shows that 2519 had no penalty imposed, 1152 unpaid, 853 paid in full; 62 missing data; 58 overpaid/adjustment; 19 partially paid

In [ ]:
civil_summons_2021['O_Penalty_Imposed'].value_counts()

In [ ]:
civil_summons_2021['O_Paid_Amount'].value_counts()

In [ ]:

civil_summons_2021['O_Penalty_Imposed'] = pd.to_numeric(
    civil_summons_2021['O_Penalty_Imposed']
        .astype(str)
        .str.replace(r'[$,]', '', regex=True)
        .str.strip()
        .replace({'': None, '.': None}),
    errors='coerce'
)

civil_summons_2021['O_Paid_Amount'] = pd.to_numeric(
    civil_summons_2021['O_Paid_Amount']
        .astype(str)
        .str.replace(r'[$,]', '', regex=True)
        .str.strip()
        .replace({'': None, '.': None}),
    errors='coerce'
)


In [ ]:
civil_summons_2021['O_Penalty_Imposed']

In [ ]:
civil_summons_2021['Amount_Difference'] = (
    civil_summons_2021['O_Penalty_Imposed'] - civil_summons_2021['O_Paid_Amount']
)
civil_summons_2021['Amount_Difference']

In [ ]:
# There are 14 tickets that Oath has not accounted for
civil_summons_2021['O_Paid_Amount'].isna().sum()

In [ ]:
# Function for binning into paid categories
def payment_status(row):
    if pd.isna(row['O_Penalty_Imposed']) or pd.isna(row['O_Paid_Amount']):
        return "Missing Data"
    if row['O_Penalty_Imposed'] == 0:
        return "No Penalty Imposed"
    if row['O_Paid_Amount'] == 0:
        return "Unpaid"
    if row['O_Paid_Amount'] == row['O_Penalty_Imposed']:
        return "Paid in Full"
    if row['O_Paid_Amount'] < row['O_Penalty_Imposed']:
        return "Partially Paid"
    if row['O_Paid_Amount'] > row['O_Penalty_Imposed']:
        return "Overpaid / Adjustment"
        
civil_summons_2021['O_Payment_Status'] = civil_summons_2021.apply(payment_status, axis=1)

In [ ]:
civil_summons_2021['O_Payment_Status'].value_counts()

#### O_Compliance_Status: Indicates whether the respondent has taken measures to address the summons
ANALYSIS: 3369 have all terms met; 1289 have a penalty due; 5 have both due

In [ ]:
civil_summons_2021['O_Compliance_Status'].value_counts()

#### O_Final_Charge <- 

#### O_Charge_Num1_Code: The OATH Code for the first charge noted
#### O_Charge_Num2_Code: The OATH Code for the second charge noted
#### O_Charge_Num3_Code: The OATH Code for the third charge noted


ANALYSIS: Some respondents have no first charge but a second charge: 3,863 codes in the first charge, 611 codes in the second charge; 25 charges in the third charge. Total 226 tickets without charges. Compiled all three to create a "Final_Charge" column. Those with multiple charges are compiled into a string list in Final_Charge. 

In [ ]:
# Charge 1 Violation 
civil_summons_2021.O_Charge_Num1_Code.count()

In [ ]:
# How many codes are in each charge
# O_Charge_Num1_Code = has the most charges
civil_summons_2021[['O_Charge_Num1_Code', 'O_Charge_Num2_Code', 'O_Charge_Num3_Code']].replace('.', pd.NA).notna().sum()

In [ ]:
# Distribution of O_Charge_Num1_Code_Description
# 1,422 citations for Failure to Yield, 804 for Open Container Consumption of Alcohol on Streets 

civil_summons_2021.O_Charge_Num1_Code_Description.value_counts(dropna=False).head(25)

In [ ]:
# Distribution of O_Charge_Num2_Code_Description
civil_summons_2021.O_Charge_Num2_Code_Description.value_counts(dropna=False).head(25)

In [ ]:
civil_summons_2021.O_Charge_Num3_Code_Description.value_counts(dropna=False).head(25)

In [ ]:
charge_cols = [
    'O_Charge_Num1_Code_Description',
    'O_Charge_Num2_Code_Description',
    'O_Charge_Num3_Code_Description'
]

def combine_all_charges(row):
    charges = [
        str(row[col]) for col in charge_cols 
        if pd.notna(row[col]) and str(row[col]).strip() != "."
    ]
    return "; ".join(charges) if charges else "."

civil_summons_2021['O_Final_Charge'] = civil_summons_2021.apply(combine_all_charges, axis=1)

In [ ]:
civil_summons_2021['O_Final_Charge'].value_counts()

In [ ]:
no_charge = (civil_summons_2021['O_Final_Charge'] == ".").sum()
print(f"Tickets with no charge description: {no_charge}")
print(f"As % of total: {no_charge / len(civil_summons_2021) * 100:.2f}%")

#### DA_charge: Description of disciplinary charge against PO
#### DA_disposition: Description of disciplinary charge against PO

#### CC_charge: Description of disciplinary charge against PO
#### CC_disposition: Description of disciplinary charge against PO

ANALYSIS: Overlap; Necessary in assessing officer behavior. Created a final_discipline_charge and final_discipline_disp column. Is this necessary for analysis when we have CCRB FADO? This is not CCRB, but nype 

In [ ]:
civil_summons_2021['DA_charge'].value_counts()

In [ ]:
civil_summons_2021['CC_charge'].value_counts()

In [ ]:
civil_summons_2021['DA_disposition'].value_counts()

In [ ]:
civil_summons_2021['CC_disposition'].value_counts()

In [ ]:
civil_summons_2021['DA_penalty'].value_counts()

In [ ]:
# Making final_discipline_charge
charge_cols = ['DA_charge', 'CC_charge']

def combine_all_charges(row):
    charges = []

    for col in charge_cols:
        value = row[col]
        if value not in [".", None, "", " "] and pd.notna(value):
            charges.append(value.strip())

    # remove duplicates, preserve order
    charges = list(dict.fromkeys(charges))

    # return "." if no real charges
    if len(charges) == 0:
        return "."

    return "; ".join(charges)

civil_summons_2021['Full_Charge'] = civil_summons_2021.apply(combine_all_charges, axis=1)


In [ ]:
civil_summons_2021['Full_Charge'].value_counts()

In [ ]:
# Category Bins fo Diciplinary History 

[
 "Excessive Force",
 "Abuse of Authority",
 "Neglect of Duty",
 "False Statements / Records",
 "Unfit for Duty / Alcohol",
 "Firearms Misconduct",
 "Misuse of Resources / Time",
 "Off-Duty Misconduct (Non-Force)"
]


#### First Set: All deleted; no relevance for analysis, duplicate variables
- FS_first_name 
- FS_middle_name
- FS_last_name
- FS_shield 


#### Disciplinary History
    -KEPT: DH_charge_count (Number of disciplinary charges made against PO), DH_allegation_count(Number of disciplinary allegations made against PO)
    -DELETED: DH_date

In [ ]:
civil_summons_2021.DH_charge_count.info()

In [ ]:
civil_summons_2021['DH_charge_count'].value_counts().sort_index()

In [ ]:
officer_charge_dist = civil_summons_2021.groupby("Final_Officer_Tax_ID")["DH_charge_count"].sum()

In [ ]:
officer_charge_dist.describe()


In [ ]:
officer_charge_dist.sort_values(ascending=False).head(20)

In [ ]:
civil_summons_2021['CC_charge'].value_counts().sort_index()

In [ ]:
civil_summons_2021.info()

#### aa_felony_arrest_count
#### aa_infraction_arrest_count
#### aa_misdemeanor_arrest_count
#### aa_other_arrest_count
#### aa_violation_arrest_count
ANALYSIS: Necessary numeric indicator for behavior/enforcement style. Some officer have 1,380 misdemeanors arrest, 229 felony arrests, 248 violation arrests, 49 infraction arrests, and 11 other arrests. 

In [ ]:
arrest_cols = [
    'aa_felony_arrest_count',
    'aa_infraction_arrest_count',
    'aa_misdemeanor_arrest_count',
    'aa_other_arrest_count',
    'aa_violation_arrest_count'
]

civil_summons_2021[arrest_cols].describe().T

#### AR_award_date: not neccessary for analysis 
#### AR_award_type

**KEEP** 

ANALYSIS: Award date is unnecessary for analysis. There are 14 types of awards: 
- COMMENDATION
- COMMENDATION-COMMUNITY SERVICE
- COMMENDATION-INTEGRITY
- EXCELLENT POLICE DUTY
- EXCEPTIONAL MERIT
- HONORABLE MENTION
- MEDAL FOR MERIT
- MEDAL FOR VALOR
- MEDAL OF HONOR
- MERITORIOUS POLICE DUTY
- MERITORIOUS POLICE DUTY-INTEGRITY
- POLICE COMBAT CROSS
- PURPLE SHIELD MEDAL 
- UNIT CITATION 

However, our officers only hold 5 awards: 
- EXCELLENT POLICE DUTY                5642
- MERITORIOUS POLICE DUTY              1950
- COMMENDATION                           77
- MERITORIOUS POLICE DUTY-INTEGRITY       3
- EXCEPTIONAL MERIT                       1

Bin these down into 4 categoris for dummy variables: 
- "commendation_award_count",
- "excellent_award_count",
- "meritorious_award_count",
- "meritorious_integrity_award_count",
- "exceptional_merit_award_count"

Added master_dept_recognitions -> total_award_count 


In [ ]:
civil_summons_2021["AR_award_type"].value_counts()

In [ ]:
# Fix the format 
import ast

def ensure_list(x):
    if x == ".":
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except:
        return []
        
civil_summons_2021["AR_award_type"] = civil_summons_2021["AR_award_type"].apply(ensure_list)


In [ ]:
# Check distribution of awards in our dataset
# There's only 5 our of the 14 awards shown for our officers
df_awards = civil_summons_2021.copy()

df_awards["AR_award_type"] = df_awards["AR_award_type"].apply(
    lambda x: x if len(x) > 0 else None
)

# Explode into one row per award
exploded = df_awards.explode("AR_award_type")

# Count awards
award_counts = exploded["AR_award_type"].value_counts()

award_counts


In [ ]:
exploded = df_awards.explode("AR_award_type")


In [ ]:
award_counts = exploded["AR_award_type"].value_counts()

award_counts

In [ ]:
# Making new columns per award
award_column_map = {
    "COMMENDATION": "commendation_award_count",
    "EXCELLENT POLICE DUTY": "excellent_award_count",
    "MERITORIOUS POLICE DUTY": "meritorious_award_count",
    "MERITORIOUS POLICE DUTY-INTEGRITY": "meritorious_integrity_award_count",
    "EXCEPTIONAL MERIT": "exceptional_merit_award_count"
}

# Create new count columns
for award, col_name in award_column_map.items():
    civil_summons_2021[col_name] = civil_summons_2021["AR_award_type"].apply(
        lambda lst: lst.count(award)
    )


In [ ]:
# list all award count columns
award_count_cols = [
    "commendation_award_count",
    "excellent_award_count",
    "meritorious_award_count",
    "meritorious_integrity_award_count",
    "exceptional_merit_award_count"
]

# make table of totals
award_summary_table = civil_summons_2021[award_count_cols].sum().reset_index()

# rename columns
award_summary_table.columns = ["award_type", "total_count"]

award_summary_table



In [ ]:
civil_summons_2021[["Final_Officer_Tax_ID"] + award_count_cols]


In [ ]:
#### master_dept_recognitions -> total_award_count 

civil_summons_2021 = civil_summons_2021.rename(
    columns={"master_dept_recognitions": "total_award_count"}
)


In [ ]:
# Checking rename
civil_summons_2021.total_award_count.head(10)

#### AD Allegations : we are using CCRB allegation data for this analysis / could categorize these for later analysis
**DELETED** 


#### master_command: PO command derived from First Set, Personnel File and Discplinary Actions with preference given to First Set for dissimilar commands listed

**DELETED & FIXED**

ANALYSIS: Use this


In [ ]:
civil_summons_2021["master_command"].value_counts().head(50)

#### master_rank: PO most recent rank derived from First Set, Personnel File and Discplinary Actions with preference given to the rank attached to the most recent date

**DELETED & FIXED**

ANALYSIS: 4,270 police officers, 230 detectives, 125 seargants, 35 unknown, and 1 captain

In [ ]:
civil_summons_2021["master_rank"].value_counts().head()

#### aa_total_arrests <- : PO total arrests as of most recent web scraping. Derived from First Set and Personnel File with preference given to the First Set but Personnel File will be used if there are no arrests in First Set

**KEPT**

ANALYSIS: 4,270 police officers, 230 detectives, 125 seargants, 35 unknown, and 1 captain

In [ ]:
civil_summons_2021.groupby("Final_Officer_Tax_ID")["total_arrests"].describe()


In [ ]:
# Make numeric
civil_summons_2021["aa_total_arrests"] = pd.to_numeric(
    civil_summons_2021["total_arrests"], errors="coerce"
)

In [ ]:
# Density Plot
sns.kdeplot(civil_summons_2021["aa_total_arrests"], fill=True)
plt.title("Density Plot of Total Arrests")
plt.xlabel("Total Arrests")
plt.ylabel("Density")

plt.savefig('EDA_figures/Density Plot of Total Arrests.png', dpi=300, bbox_inches='tight')
plt.show()



#### final_race: PO race/ethnicity. Derived from First Set and Personnel File with preference given to the First Set but Personnel File will be used if there is no race/ethnicity listed in First Set. 
**DELETED & FIXED**


ANALYSIS: Had to directly pull from Legal AID rster; too many missing race values (35), now there are only 3 unmatched

In [ ]:
# fixing the period
civil_summons_2021["final_race"] = civil_summons_2021["final_race"].str.strip()

In [ ]:
civil_summons_2021["Final_Officer_Tax_ID"] = (
    civil_summons_2021["Final_Officer_Tax_ID"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

civil_summons_2021.loc[civil_summons_2021["Final_Officer_Tax_ID"] == ".", "Final_Officer_Tax_ID"] = None


In [ ]:
# The 37 are not accounted for, let me grab that from the legal aid roster 
civil_summons_2021.final_race.value_counts()

In [ ]:
missing_race_df = civil_summons_2021[civil_summons_2021["final_race"] == "."]
missing_race_df.shape

In [ ]:
missing_tax_ids = missing_race_df["Final_Officer_Tax_ID"].unique().tolist()
missing_tax_ids

In [ ]:
cs_tax_ids = civil_summons_2021["Final_Officer_Tax_ID"].unique().tolist()
cs_tax_ids

### Merging Legal Aid Roster Data: 

Current.rank Map: 
"1": "CAPTAIN",
"2": "CHIEFS AND OTHER RANKS",
"3": "DEPUTY INSPECTOR",
"4": "DETECTIVE",
"5": "INACTIVE RANKS",
"6": "INSPECTOR",
"7": "LIEUTENANT",
"8": "POLICE OFFICER",
"9": "SERGEANT"

Race Map: 
"1": "AMERICAN INDIAN",
"2": "ASIAN",
"3": "BLACK",
"4": "HISPANIC",
"5": "OTHER RACE",
"6": "WHITE"

Gender Map: 
"1": "FEMALE",
"2": "MALE",
"3": "TGNC / OTHER"


ANALYSIS: Use these rank, command, race, gender, instead of the presious ones. Previously 35 officers were unaccounted for, now this merge with the most active roster 3 officer are unaccounted for. 

In [ ]:
# There was a missing 35 from FS and PF, I directly pulled from Legal Aid NYPD Roster to get the most accurate and filled data
# I want Current.rank, Current.command, Race, Gender
roster_2021 = pd.read_csv(
    "data/roster.tsv",
    sep="\t",
    dtype=str
)

# attach to see how many internal misconduct record with our officers 
matched_ids = roster_2021[roster_2021["Tax.."].isin(cs_tax_ids)]

In [ ]:
len(cs_tax_ids)

In [ ]:
len(matched_ids)

In [ ]:
matched_taxids = matched_ids["Tax.."].unique().tolist()


In [ ]:
# Great. All of our officers 
unmatched_ids = sorted(list(set(cs_tax_ids) - set(matched_taxids)))
unmatched_ids


In [ ]:
roster_2021.columns.to_list()

In [ ]:
matched_ids.columns

In [ ]:
# what I want from roster (Current.rank, Current.command, Race, Gender) 
roster_subset = matched_ids[[
    'Tax..',
    'Current.rank',
    'Current.command',
    'Race',
    'Gender',
    'Total.CCRB.complaints',
    'Total.substantiated.CCRB.complaints', 
    'Last.active',
    'Officer.first.name', 
    'Officer.last.name',
    'AKA'
]].copy()

roster_subset = roster_subset.rename(columns={"Tax..": "Final_Officer_Tax_ID"})


In [ ]:
# Merge into civil summons
civil_summons_2021 = civil_summons_2021.merge(
    roster_subset,
    on="Final_Officer_Tax_ID",
    how="left"
)

In [ ]:
# Checking 
civil_summons_2021[[
    "Final_Officer_Tax_ID",
    "Current.rank",
    "Current.command",
    "Race",
    "Gender",
    "Total.CCRB.complaints",
    "Total.substantiated.CCRB.complaints", 
    "Last.active",
    "Officer.first.name", 
    "Officer.last.name",
    "AKA"
]].head()


In [ ]:
# Checking merge 
cs_ids = set(civil_summons_2021["Final_Officer_Tax_ID"].unique())
roster_ids = set(roster_2021["Tax.."].unique())

matched = cs_ids & roster_ids
unmatched = cs_ids - roster_ids

print("Total civil summons officers:", len(cs_ids))
print("Matched in roster:", len(matched))
print("NOT matched:", len(unmatched))


In [ ]:
# OFFICER DEMOGRAPHICS: 1956 WHITE, 1397 HISPANIC, 659 ASIAN, 646 BLACK; 5 OTHER RACE 

civil_summons_2021.Race.value_counts()

In [ ]:
# OFFICER DEMOGRAPHICS: 3795 MALE, 864 FEMALE, 3 TGNC/OTHER

civil_summons_2021.Gender.value_counts()

In [ ]:
# OFFICER RANK: 3403 POLICE OFFICERS, 677 SERGEANT, 575 DETECTIVES, 6 LIEUTENANTS, 1 CAPTAIN, 1 INSPECTOR

civil_summons_2021['Current.rank'].value_counts()

In [ ]:
# Checking missing. None missing. 
known_rank_officers = civil_summons_2021[civil_summons_2021["Current.rank"].notna()]["Final_Officer_Tax_ID"].nunique()
known_rank_officers


In [ ]:
# Renaming columns
civil_summons_2021 = civil_summons_2021.rename(
    columns={
        "Current.rank": "roster_rank",
        "Current.command": "roster_command",
        "Race": "roster_race", 
        "Gender": "roster_gender"
        
    }
)

In [ ]:
civil_summons_2021.roster_race.value_counts()

### Merging Legal Aid Payroll Data 
Match via roster Officer.first.name and Officer.last.name

- Years of Service (created from Start.date) 
- Reg.hours
- Overtime.hours

In [ ]:
civil_summons_2021['Final_Officer_First_Name'].head()

In [ ]:
civil_summons_2021.columns.tolist()

In [ ]:
# Add Middle.init from AKA to help differentiate officer 
civil_summons_2021['Middle.init'] = (
    civil_summons_2021['AKA']
    .str.extract(r'\b([A-Z])$')  # last capital letter
)

In [ ]:
# Load in payroll fiscal year 2021
payroll_2021 = pd.read_csv("data/payroll_2020_2021 (1).csv")
payroll_2021 = payroll_2021[payroll_2021['Fiscal.year'] == 2021]
payroll_2021 = payroll_2021[payroll_2021['Agency'] == 'POLICE DEPARTMENT']
payroll_2021.to_csv("manual_payroll_merge_data/current_payroll.csv")

In [ ]:
payroll_2021.head()

In [ ]:
#payroll_2021['first']  = payroll_2021['First.name']
#payroll_2021['last']   = payroll_2021['Last.name']
#payroll_2021['middle'] = payroll_2021['Middle.init']

In [ ]:
#civil_summons_2021['first']  = civil_summons_2021['Officer.first.name']
#civil_summons_2021['last']   = civil_summons_2021['Officer.last.name']
#civil_summons_2021['middle'] = civil_summons_2021['Middle.init']

In [ ]:
#officers = civil_summons_2021[['first','last','middle','Final_Officer_Tax_ID']].drop_duplicates()
#len(officers)

In [ ]:
#payroll_with_ids = payroll_2021.merge(
#    officers,
#    on=['first','last','middle'],
#    how='inner'
#)

In [ ]:
#len(payroll_with_ids)


In [ ]:
len(payroll_2021)

In [ ]:
payroll_2021.info()

In [ ]:

#trial 
def clean_name(s):
    return (
        s.astype(str)
         .str.upper()
         .str.replace(r'[^A-Z]', '', regex=True)
    )

civil_summons_2021['cs_first']  = clean_name(civil_summons_2021['Officer.first.name'])
civil_summons_2021['cs_last']   = clean_name(civil_summons_2021['Officer.last.name'])
civil_summons_2021['cs_middle'] = clean_name(civil_summons_2021['Middle.init'])

# Payroll
payroll_2021['pr_first']  = clean_name(payroll_2021['First.name'])
payroll_2021['pr_last']   = clean_name(payroll_2021['Last.name'])
payroll_2021['pr_middle'] = clean_name(payroll_2021['Middle.init'])

In [ ]:
civil_summons_2021.head()

In [ ]:
# Standardize start date
payroll_2021['Start.date'] = (
    pd.to_datetime(payroll_2021['Start.date'], errors='coerce')
      .dt.date
      .astype(str)
)

In [ ]:
civil_names = (
    civil_summons_2021[['cs_first', 'cs_last', 'cs_middle', 'Final_Officer_Tax_ID']]
    .drop_duplicates()
)

payroll_names = (
    payroll_2021[['pr_first', 'pr_last', 'pr_middle']]
    .drop_duplicates()
)


In [ ]:
exact_matches = civil_names.merge(
    payroll_names,
    left_on=['cs_first', 'cs_last', 'cs_middle'],
    right_on=['pr_first', 'pr_last', 'pr_middle'],
    how='inner'
)

In [ ]:
exact_matches

In [ ]:
# Out of 2423 officers in CS, only 2162 officers match with payroll data. 
len(exact_matches)

In [ ]:
# Lets look at these 2161 matches 
payroll_match_counts = (
    payroll_2021
    .merge(
        exact_matches,
        left_on=['pr_first', 'pr_last', 'pr_middle'],
        right_on=['pr_first', 'pr_last', 'pr_middle'],
        how='inner'
    )
    .groupby(['pr_first', 'pr_last', 'pr_middle'])
    .size()
    .reset_index(name='payroll_row_count')
)
payroll_match_counts

In [ ]:
# Out of the 2161 matches, 2133 appear once in payroll, so we can safely merge associated tax id with these officers 
safe_exact_matches = payroll_match_counts[
    payroll_match_counts['payroll_row_count'] == 1
]
len(safe_exact_matches)

In [ ]:
### First Merge ### 
safe_tax_lookup = (
    civil_summons_2021
    .merge(
        safe_exact_matches,
        left_on=['cs_first', 'cs_last', 'cs_middle'],
        right_on=['pr_first', 'pr_last', 'pr_middle'],
        how='inner'
    )
    [['cs_first', 'cs_last', 'cs_middle', 'Final_Officer_Tax_ID']]
    .drop_duplicates()
)

payroll_2021 = payroll_2021.merge(
    safe_tax_lookup,
    left_on=['pr_first', 'pr_last', 'pr_middle'],
    right_on=['cs_first', 'cs_last', 'cs_middle'],
    how='left'
)

payroll_2021 = payroll_2021.drop(
    columns=['cs_first', 'cs_last', 'cs_middle'],
    errors='ignore'
)


In [ ]:
payroll_2021['Final_Officer_Tax_ID'].notna().sum()


In [ ]:
payroll_2021['Final_Officer_Tax_ID'].nunique()
# Great, a perfect merge. 

In [ ]:
# These are exact matches across name, but appear 2x in payroll - review and differentiate based on shield num. 
ambiguous_exact_matches = payroll_match_counts[
    payroll_match_counts['payroll_row_count'] > 1
]
len(ambiguous_exact_matches)

In [ ]:
ambiguous_exact_matches.to_csv("manual_payroll_merge_data/ambiguous_exact_matches.csv")

In [ ]:
# Fully investigated - matched Start.date by shield num associated to taxIO
ambiguous_exact_matches_resolved = pd.read_csv("manual_payroll_merge_data/ambiguous_exact_matches_resolved.csv")

In [ ]:
ambiguous_exact_matches_resolved.head()

In [ ]:
ambiguous_exact_matches_resolved['Start.date_found_byshield'] = (
    pd.to_datetime(ambiguous_exact_matches_resolved['Start.date_found_byshield'], errors='coerce')
      .dt.date
      .astype(str)
)

In [ ]:
# Fix NAs 
ambiguous_exact_matches_resolved['pr_middle'] = (
    ambiguous_exact_matches_resolved['pr_middle']
        .fillna('')
        .astype(str)
)
ambiguous_exact_matches_resolved['pr_middle'].head()

In [ ]:
### Second Merge w/28 ###
# Merge the 28 resolved , we should get our original 2161 exact matches now

ambig_resolver = ambiguous_exact_matches_resolved.rename(
    columns={'Start.date_found_byshield': 'Start.date'}
)

# Identify the correct payroll rows
payroll_ambig_matched = payroll_2021.merge(
    ambig_resolver[['pr_first', 'pr_last', 'pr_middle', 'Start.date']],
    on=['pr_first', 'pr_last', 'pr_middle', 'Start.date'],
    how='inner'
)

cs_tax_lookup = (
    civil_summons_2021[
        ['cs_first', 'cs_last', 'cs_middle', 'Final_Officer_Tax_ID']
    ]
    .drop_duplicates()
)

payroll_ambig_with_ids = payroll_ambig_matched.merge(
    cs_tax_lookup,
    left_on=['pr_first', 'pr_last', 'pr_middle'],
    right_on=['cs_first', 'cs_last', 'cs_middle'],
    how='left',
    suffixes=('', '_cs')
)

payroll_2021 = payroll_2021.merge(
    payroll_ambig_with_ids[
        ['pr_first', 'pr_last', 'pr_middle', 'Start.date', 'Final_Officer_Tax_ID_cs']
    ],
    on=['pr_first', 'pr_last', 'pr_middle', 'Start.date'],
    how='left',
    suffixes=('', '_ambig')
)

# Fill only missing IDs
payroll_2021['Final_Officer_Tax_ID'] = (
    payroll_2021['Final_Officer_Tax_ID']
    .combine_first(payroll_2021['Final_Officer_Tax_ID_cs'])
)

payroll_2021.drop(columns='Final_Officer_Tax_ID_cs', inplace=True)


In [ ]:
# Perfect, now we have 2161 officers classified with tax id. 
payroll_2021['Final_Officer_Tax_ID'].notna().sum()

In [ ]:
# There are 261 officers thats whole name is not in payroll - these are mostly those who have mearried and have a changed last name, different name spelli
# spellings, etc. 
civil_only = civil_names.merge(
    payroll_names,
    left_on=['cs_first', 'cs_last', 'cs_middle'],
    right_on=['pr_first', 'pr_last', 'pr_middle'],
    how='left',
    indicator=True
).query("_merge == 'left_only'")


In [ ]:
len(civil_only)

In [ ]:
# Manually investigate - majority of these names are those whose last name changed from marriage, JR, II excluded 
civil_only.to_csv("manual_payroll_merge_data/civil_only.csv")

In [ ]:
civil_only.head()

In [ ]:
civil_only_resolved = pd.read_csv("manual_payroll_merge_data/civil_only_resolved.csv")
civil_only_resolved.head()

In [ ]:
len(civil_only_resolved)

In [ ]:

civil_only_resolved.columns.to_list()

In [ ]:
civil_only_resolved['cs_first']  = clean_name(civil_only_resolved['cs_first'])
civil_only_resolved['cs_last']   = clean_name(civil_only_resolved['cs_last'])
civil_only_resolved['cs_middle']   = clean_name(civil_only_resolved['cs_middle'])
civil_only_resolved['updated_pr_last']  = clean_name(civil_only_resolved['updated_pr_last'])
civil_only_resolved['updated_pr_first']   = clean_name(civil_only_resolved['updated_pr_first'])
civil_only_resolved['updated_pr_middle']   = clean_name(civil_only_resolved['updated_pr_middle'])

In [ ]:
# Standarize NAs
#missing_strings = ['NAN', 'NaN', 'nan', 'None', 'NULL']

#civil_only_resolved = (
#    civil_only_resolved
#    .replace(missing_strings, '')
#    .fillna('')
#)

civil_only_resolved = civil_only_resolved.replace(
    ['nan', 'NAN', 'NaN', '', 'None', 'NONE'],
    np.nan
)

In [ ]:
civil_only_resolved.head()

In [ ]:
# How many should we really look out for? 155
civil_only_resolved['no_info_flag'] = (
    civil_only_resolved[['updated_pr_first', 'updated_pr_last', 'last_active', 'condition']]
        .replace('', pd.NA)
        .isna()
        .all(axis=1)
)
civil_only_resolved['no_info_flag'].value_counts()

In [ ]:
# How many merge based on condition? 12
civil_only_resolved['condition'].notna().sum()

In [ ]:
# How many ids have a last active date? 25
civil_only_resolved['last_active'].notna().sum()

In [ ]:
# How many ids have a pr_last? 109
civil_only_resolved['updated_pr_last'].notna().sum()

In [ ]:
# How many ids have a pr_first? 11
civil_only_resolved['updated_pr_first'].notna().sum()

In [ ]:
# How many ids have pr_first and pr_last? 1
(
    (civil_only_resolved['updated_pr_first'].notna()) &
    (civil_only_resolved['updated_pr_last'].notna())
).sum()


In [ ]:
civil_only_resolved.columns.to_list()

In [ ]:
civil_only_resolved.head()

In [ ]:
payroll_2021.head()

In [ ]:
payroll_2021 = payroll_2021.replace(
    ['nan', 'NAN', 'NaN', '', 'None', 'NONE'],
    np.nan
)

In [ ]:
payroll_2021.head()

In [ ]:
resolver = (
    civil_only_resolved[
        ['cs_first', 'updated_pr_last', 'Final_Officer_Tax_ID']
    ]
    .drop_duplicates()
    .rename(columns={
        'cs_first': 'pr_first',
        'updated_pr_last': 'pr_last'
    })
)

payroll_2021 = payroll_2021.merge(
    resolver,
    on=['pr_first', 'pr_last'],
    how='left',
    suffixes=('', '_resolved')
)

payroll_2021['Final_Officer_Tax_ID'] = (
    payroll_2021['Final_Officer_Tax_ID']
        .combine_first(payroll_2021['Final_Officer_Tax_ID_resolved'])
)


payroll_2021.drop(columns='Final_Officer_Tax_ID_resolved', inplace=True)

In [ ]:
# Got 108/109 of those with last names to merge from 2161 
payroll_2021['Final_Officer_Tax_ID'].notna().sum()

In [ ]:
# Another merge based on this group: 
resolver_2 = (
    civil_only_resolved[
        ['cs_last', 'updated_pr_first', 'Final_Officer_Tax_ID']
    ]
    .drop_duplicates()
    .rename(columns={
        'cs_last': 'pr_last',
        'updated_pr_first': 'pr_first'
    })
)

payroll_2021 = payroll_2021.merge(
    resolver_2,
    on=['pr_first', 'pr_last'],
    how='left',
    suffixes=('', '_resolved2')
)

payroll_2021['Final_Officer_Tax_ID'] = (
    payroll_2021['Final_Officer_Tax_ID']
        .combine_first(payroll_2021['Final_Officer_Tax_ID_resolved2'])
)

payroll_2021.drop(columns='Final_Officer_Tax_ID_resolved2', inplace=True)


In [ ]:
# Got all of the 11 changed 
payroll_2021['Final_Officer_Tax_ID'].notna().sum()

In [ ]:
civil_only_resolved.condition.value_counts()

In [ ]:
# Merge the rest by found condition
civil_only_resolved['condition'] = (
    pd.to_datetime(civil_only_resolved['condition'], errors='coerce')
      .dt.date
      .astype(str)
)

In [ ]:
resolver_condition = (
    civil_only_resolved[
        ['cs_first', 'cs_last', 'condition', 'Final_Officer_Tax_ID']
    ]
    .dropna(subset=['cs_first', 'cs_last', 'condition', 'Final_Officer_Tax_ID'])
    .drop_duplicates()
)

payroll_2021 = payroll_2021.merge(
    resolver_condition,
    left_on=['pr_first', 'pr_last', 'Start.date'],
    right_on=['cs_first', 'cs_last', 'condition'],
    how='left',
    suffixes=('', '_cond')
)

payroll_2021['Final_Officer_Tax_ID'] = (
    payroll_2021['Final_Officer_Tax_ID']
        .combine_first(payroll_2021['Final_Officer_Tax_ID_cond'])
)

payroll_2021.drop(
    columns=['Final_Officer_Tax_ID_cond', 'condition'],
    inplace=True,
    errors='ignore'
)


In [ ]:
# Got all of the 10 changed 
payroll_2021['Final_Officer_Tax_ID'].notna().sum()

In [ ]:
#if there are values in the condition colum and there are no duplicate sets in payroll_2021 of cs_last and cs_first sets == pr_last and pr_first 
# (whether there is a value in cs_middle/pr_middle or not), match the final_officer_tax id to payroll_2021

unique_payroll_names = (
    payroll_2021
        .groupby(['pr_first', 'pr_last'])
        .size()
        .reset_index(name='n')
        .query('n == 1')
        [['pr_first', 'pr_last']]
)

civil_name_resolver = (
    civil_only_resolved
        .loc[
            civil_only_resolved['condition'].isna() |
            (civil_only_resolved['condition'] == ''),
            ['cs_first', 'cs_last', 'Final_Officer_Tax_ID']
        ]
        .drop_duplicates()
)


safe_resolver = civil_name_resolver.merge(
    unique_payroll_names,
    left_on=['cs_first', 'cs_last'],
    right_on=['pr_first', 'pr_last'],
    how='inner'
)


payroll_2021 = payroll_2021.merge(
    safe_resolver[['pr_first', 'pr_last', 'Final_Officer_Tax_ID']],
    on=['pr_first', 'pr_last'],
    how='left',
    suffixes=('', '_uniq')
)


payroll_2021['Final_Officer_Tax_ID'] = (
    payroll_2021['Final_Officer_Tax_ID']
        .combine_first(payroll_2021['Final_Officer_Tax_ID_uniq'])
)

payroll_2021.drop(columns='Final_Officer_Tax_ID_uniq', inplace=True)



In [ ]:
payroll_2021['Final_Officer_Tax_ID'].notna().sum() #2388

In [ ]:
len(safe_resolver)

In [ ]:
civil_only_resolved.nunique()

In [ ]:
# Are the name cs_first and cs_last name pairings in civil_only_resolved duplicates in payroll_2021 

In [ ]:
civil_pairs = (
    civil_only_resolved[['cs_first', 'cs_last']]
    .drop_duplicates()
)
len(civil_pairs)

In [ ]:
payroll_name_counts = (
    payroll_2021
        .groupby(['pr_first', 'pr_last'])
        .size()
        .reset_index(name='payroll_count')
)
civil_in_payroll_counts = civil_pairs.merge(
    payroll_name_counts,
    left_on=['cs_first', 'cs_last'],
    right_on=['pr_first', 'pr_last'],
    how='left'
)


In [ ]:
duplicated_name_pairs = civil_in_payroll_counts[
    civil_in_payroll_counts['payroll_count'] > 1
]
duplicated_name_pairs

In [ ]:
civil_only_resolved.condition.value_counts()

In [ ]:
# Unique IDs in civil summons (ground truth universe)
civil_ids = (
    civil_summons_2021['Final_Officer_Tax_ID']
    .dropna()
    .unique()
)

# Unique IDs successfully attached to payroll
payroll_ids = (
    payroll_2021['Final_Officer_Tax_ID']
    .dropna()
    .unique()
)


In [ ]:
# These are those that could not be determines with a merging rule 
civil_not_in_payroll = sorted(
    set(civil_ids) - set(payroll_ids)
)

len(civil_not_in_payroll)


In [ ]:
civil_not_in_payroll_named = (
    civil_summons_2021[
        civil_summons_2021['Final_Officer_Tax_ID']
            .astype(str)
            .isin(civil_not_in_payroll)
    ][
        ['Final_Officer_Tax_ID', 'Officer.first.name', 'Officer.last.name', 'Middle.init']
    ]
    .drop_duplicates()
)


In [ ]:
# Look at these and determine a rule
# Merge based on whether pahyroll_2021 has first name AND last name AND Start.date. This unit is the key to 
civil_not_in_payroll_named.to_csv("manual_payroll_merge_data/last_ids_merged.csv")

In [ ]:
payroll_2021.columns.to_list()

In [ ]:

civil_not_in_payroll_named.head()

In [ ]:
# Now resolved last ids 
last_ids_resolved = pd.read_csv("manual_payroll_merge_data/last_ids_merged_resolved.csv")

In [ ]:
len(last_ids_resolved)

In [ ]:
last_ids_resolved.head()

In [ ]:
# Standardize these
last_ids_resolved['update_pr_last']  = clean_name(last_ids_resolved['update_pr_last'])
last_ids_resolved['updated_pr_first']   = clean_name(last_ids_resolved['updated_pr_first'])

In [ ]:
# Standardize Updated.Start.Date 
last_ids_resolved['Updated.Start.Date'] = (
    pd.to_datetime(last_ids_resolved['Updated.Start.Date'], errors='coerce')
      .dt.date
      .astype(str)
)

In [ ]:
payroll_2021.info()

In [ ]:
last_ids_resolved.columns.to_list()

In [ ]:
# We need these 236 to merge 
(last_ids_resolved['In_Payroll?'] == 'Y').sum()

In [ ]:
# These are 
(last_ids_resolved['In_Payroll?'] == 'N').sum()

In [ ]:
no_connection_ids = last_ids_resolved[last_ids_resolved['In_Payroll?'] == 'N']

In [ ]:
last_ids_resolved = last_ids_resolved[last_ids_resolved['In_Payroll?'] == 'Y']

In [ ]:
payroll_2021.head()

In [ ]:
last_ids_resolved.head()

In [ ]:

payroll_2021['Start.date'].head()

In [ ]:
def clean_name_col(s):
    return (
        s
        .astype(str)
        .where(s.notna())  # restore NaN after astype
        .str.upper()
        .str.strip()
        .str.replace(r'[^A-Z]', '', regex=True)
    )

def clean_date_col(s):
    return (
        pd.to_datetime(s, errors='coerce')
          .dt.date
          .astype(str)
    )

def clean_id_col(s):
    return (
        s.dropna()
         .astype(str)
         .str.replace(r'\.0$', '', regex=True)
    )


In [ ]:
# Names
payroll_2021['pr_first']  = clean_name_col(payroll_2021['pr_first'])
payroll_2021['pr_last']   = clean_name_col(payroll_2021['pr_last'])
payroll_2021['pr_middle'] = clean_name_col(payroll_2021['pr_middle'])
 
# Dates
payroll_2021['Start.date'] = clean_date_col(payroll_2021['Start.date'])

# IDs
payroll_2021['Final_Officer_Tax_ID'] = (
    payroll_2021['Final_Officer_Tax_ID']
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
)
payroll_2021['Final_Officer_Tax_ID'] = (
    payroll_2021['Final_Officer_Tax_ID']
        .replace(['nan', 'NaN', '', 'None'], np.nan)
)

In [ ]:
# Names
last_ids_resolved['updated_pr_first']  = clean_name_col(last_ids_resolved['updated_pr_first'])
last_ids_resolved['update_pr_last']   = clean_name_col(last_ids_resolved['update_pr_last'])

# Dates
last_ids_resolved['Updated.Start.Date'] = clean_date_col(
    last_ids_resolved['Updated.Start.Date']
) 

# IDs
last_ids_resolved['Final_Officer_Tax_ID'] = (
    last_ids_resolved['Final_Officer_Tax_ID']
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
)


In [ ]:
last_ids_resolved.head()

In [ ]:
payroll_2021.head()

In [ ]:
# No NaNs in merge keys
#assert payroll_2021[['pr_first', 'pr_last', 'Start.date']].isna().sum().sum() == 0
#assert last_ids_resolved[['updated_pr_first', 'update_pr_last', 'Updated.Start.Date']].isna().sum().sum() == 0

# Same dtypes
payroll_2021[['pr_first', 'pr_last', 'Start.date']].dtypes
last_ids_resolved[['updated_pr_first', 'update_pr_last', 'Updated.Start.Date']].dtypes


In [ ]:
#stop
#diagnostic = payroll_2021.merge(
#    last_ids_resolved,
 #   left_on=['pr_first', 'pr_last', 'Start.date'],
 #   right_on=['updated_pr_first', 'update_pr_last', 'Updated.Start.Date'],
# #   how='inner'
#)

#len(diagnostic)


In [ ]:
resolver_236 = (
    last_ids_resolved[
        ['updated_pr_first', 'update_pr_last', 'Updated.Start.Date', 'Final_Officer_Tax_ID']
    ]
    .dropna(subset=[
        'updated_pr_first',
        'update_pr_last',
        'Updated.Start.Date',
        'Final_Officer_Tax_ID'
    ])
    .drop_duplicates()
)
len(resolver_236)


In [ ]:
## Final Merge ## 
payroll_2021 = payroll_2021.merge(
    resolver_236,
    left_on=['pr_first', 'pr_last', 'Start.date'],
    right_on=['updated_pr_first', 'update_pr_last', 'Updated.Start.Date'],
    how='left',
    suffixes=('', '_resolved')
)
payroll_2021['Final_Officer_Tax_ID'] = (
    payroll_2021['Final_Officer_Tax_ID']
        .combine_first(payroll_2021['Final_Officer_Tax_ID_resolved'])
)
payroll_2021.drop(
    columns=[
        'updated_pr_first',
        'update_pr_last',
        'Updated.Start.Date'
    ],
    inplace=True,
    errors='ignore'
)


In [ ]:
payroll_2021['Final_Officer_Tax_ID'].notna().sum()

In [ ]:
civil_ids = (
    civil_summons_2021['Final_Officer_Tax_ID']
        .dropna()
        .astype(str)
        .unique()
)

payroll_ids = (
    payroll_2021['Final_Officer_Tax_ID']
        .dropna()
        .astype(str)
        .unique()
)


In [ ]:
matched_ids = set(civil_ids) & set(payroll_ids)
len(matched_ids)


In [ ]:
payroll_2021.head()

In [ ]:
civil_not_in_payroll = sorted(
    set(civil_ids) - set(payroll_ids)
)

len(civil_not_in_payroll)


In [ ]:
civil_not_in_payroll

In [ ]:
# How many of civil_not_in_payroll ids are in no_connection_ids['Final_Officer_Tax_ID'] ids, give me the remainder of the tx ids 

In [ ]:
payroll_2021.info()

In [ ]:
# 939707 - Raymond Wong 2005-07-11T00:00:00Zdid not merge 
payroll_2021.loc[
    (payroll_2021['Last.name'] == 'WONG') &
    (payroll_2021['First.name'] == 'RAYMOND') &
    (payroll_2021['Start.date'] == '2005-07-11'),
    'Final_Officer_Tax_ID'
] = 939707


In [ ]:
# 945652 DELACRUZ RAFAEL 2008-01-07T00:00:00Z - Did not merg 
payroll_2021.loc[
    (payroll_2021['Last.name'] == 'DELACRUZ') &
    (payroll_2021['First.name'] == 'RAFAEL') &
    (payroll_2021['Start.date'] == '2008-01-07'),
    'Final_Officer_Tax_ID'
] = 945652


In [ ]:
# 954276 RIVERA	RAYMOND	2013-01-09T00:00:00Z
payroll_2021.loc[
    (payroll_2021['Last.name'] == 'RIVERA') &
    (payroll_2021['First.name'] == 'RAYMOND') &
    (payroll_2021['Start.date'] == '2013-01-09'),
    'Final_Officer_Tax_ID'
] = 954276

In [ ]:
# 959988 SINGH	HARMANJOT	2015-10-07T00:00:00Z
payroll_2021.loc[
    (payroll_2021['Last.name'] == 'SINGH') &
    (payroll_2021['First.name'] == 'HARMANJOT') &
    (payroll_2021['Start.date'] == '2015-10-07'),
    'Final_Officer_Tax_ID'
] = 959988

In [ ]:
# 961189 RODRIGUEZ	CHRISTIAN	2016-01-06T00:00:00Z
payroll_2021.loc[
    (payroll_2021['Last.name'] == 'RODRIGUEZ') &
    (payroll_2021['First.name'] == 'CHRISTIAN') &
    (payroll_2021['Start.date'] == '2016-01-06'),
    'Final_Officer_Tax_ID'
] = 961189

In [ ]:
# 962080 SALERNO MICHAEL Last.Active = 2021-10-30


In [ ]:
# 965942 BAEZ	JONATHAN	2018-07-02T00:00:00Z
payroll_2021.loc[
    (payroll_2021['Last.name'] == 'BAEZ') &
    (payroll_2021['First.name'] == 'JONATHAN') &
    (payroll_2021['Start.date'] == '2018-07-02'),
    'Final_Officer_Tax_ID'
] = 965942

In [ ]:
# 965980 CHEN	ALEX	2018-07-02T00:00:00Z
payroll_2021.loc[
    (payroll_2021['Last.name'] == 'CHEN') &
    (payroll_2021['First.name'] == 'ALEX') &
    (payroll_2021['Start.date'] == '2018-07-02'),
    'Final_Officer_Tax_ID'
] = 965980

In [ ]:
# 966307 RODRIGUEZ	CHRISTIAN	2018-07-02T00:00:00Z
payroll_2021.loc[
    (payroll_2021['Last.name'] == 'RODRIGUEZ') &
    (payroll_2021['First.name'] == 'CHRISTIAN') &
    (payroll_2021['Start.date'] == '2018-07-02'),
    'Final_Officer_Tax_ID'
] = 966307

In [ ]:
# 968565 LEE	DANIEL	2019-10-07T00:00:00Z
payroll_2021.loc[
    (payroll_2021['Last.name'] == 'LEE') &
    (payroll_2021['First.name'] == 'DANIEL') &
    (payroll_2021['Start.date'] == '2019-10-07'),
    'Final_Officer_Tax_ID'
] = 968565

In [ ]:
# 969691 FERNANDEZ	JASON	2020-11-02T00:00:00Z
payroll_2021.loc[
    (payroll_2021['Last.name'] == 'FERNANDEZ') &
    (payroll_2021['First.name'] == 'JASON') &
    (payroll_2021['Start.date'] == '2020-11-02'),
    'Final_Officer_Tax_ID'
    ] = 969691

In [ ]:
# 970090 PIERRE	MIGUEL	2020-11-02T00:00:00Z
payroll_2021.loc[
    (payroll_2021['Last.name'] == 'PIERRE') &
    (payroll_2021['First.name'] == 'MIGUEL') &
    (payroll_2021['Start.date'] == '2020-11-02'),
    'Final_Officer_Tax_ID'
] = 970090

In [ ]:
# 966829 MICHAEL L. SALERNO Last.Active = 2021-10-30
payroll_2021.loc[
    (payroll_2021['Last.name'] == 'SALERNO') &
    (payroll_2021['First.name'] == 'MICHAEL') &
    (payroll_2021['Start.date'] == '2018-10-24'),
    'Final_Officer_Tax_ID'
] = 966829

In [ ]:
payroll_2021['Final_Officer_Tax_ID'].notna().sum()

In [ ]:
len(no_connection_ids)

In [ ]:
civil_ids = (
    civil_summons_2021['Final_Officer_Tax_ID']
        .dropna()
        .astype(str)
        .unique()
)

payroll_ids = (
    payroll_2021['Final_Officer_Tax_ID']
        .dropna()
        .astype(str)
        .unique()
)

matched_ids = set(civil_ids) & set(payroll_ids)
len(matched_ids)

In [ ]:
civil_not_in_payroll = sorted(
    set(civil_ids) - set(payroll_ids)
)

len(civil_not_in_payroll)

In [ ]:
# All of these and 962080 SALERNO MICHAEL Last.Active = 2021-10-30 are not found within cvil 
civil_not_in_payroll

In [ ]:
payroll_2021['Final_Officer_Tax_ID'].notna().sum()

In [ ]:

payroll_2021.columns.to_list()

In [ ]:
civil_summons_2021.columns.to_list()

In [ ]:
payroll_2021['Final_Officer_Tax_ID'] = pd.to_numeric(
    payroll_2021['Final_Officer_Tax_ID'], errors='coerce'
)

civil_summons_2021['Final_Officer_Tax_ID'] = pd.to_numeric(
    civil_summons_2021['Final_Officer_Tax_ID'], errors='coerce'
)


#### Merge payroll_2021 Reg.hours, Overtime.hours, and Start.date to civil_summons_2021

In [ ]:
payroll_merge = payroll_2021[
    [
        'Final_Officer_Tax_ID',
        'Reg.hours',
        'Overtime.hours',
        'Start.date', 
        'pr_first',
        'pr_last',
        'pr_middle'
    ]
].copy()


In [ ]:
(payroll_merge['Final_Officer_Tax_ID'] == 959988).any()

In [ ]:
len(payroll_merge)

In [ ]:
civil_summons_2021 = civil_summons_2021.merge(
    payroll_merge,
    how='left',
    on='Final_Officer_Tax_ID'
)


In [ ]:

civil_summons_2021[['Reg.hours', 'Overtime.hours']].notna().any(axis=1).mean()


In [ ]:
civil_summons_2021.head()

In [ ]:
# Missing the 27 justified, not identified IDs ; totalling to 33 citations total 
civil_summons_2021.to_csv("data/civil_summons_2021_v1.csv")


#### Transform civil_summons_2021 Start.date into Years of Service
- From Start.date 

In [ ]:
civil_summons_2021['Start.date']

In [ ]:
civil_summons_2021['Start.date'] = pd.to_datetime(
    civil_summons_2021['Start.date'],
    errors='coerce'
)

reference_date = pd.Timestamp('2026-01-09')  # or pd.Timestamp.today()

civil_summons_2021['yrs_of_service'] = (
    (reference_date - civil_summons_2021['Start.date'])
    .dt.days // 365
).astype('Int64')

In [ ]:
civil_summons_2021['yrs_of_service'].hist()

In [ ]:
civil_summons_2021['yrs_of_service'].describe()

In [ ]:
civil_summons_2021['Reg.hours'].describe()

In [ ]:
civil_summons_2021.columns.to_list()

In [ ]:
civil_summons_2021['Reg.hours'].hist()

In [ ]:
civil_summons_2021['Overtime.hours'].hist()

#### Manually associate Tax IDs for Unusual Names 
- Payroll names do not match  Roster names. This does not indicate that these officers are not found, they just have differnt associated names across these two databases. However, we can still attach a Tax ID to these names by looking at:
  - Roster AKA names (usually contains married changed names)
  - Shield number associated to Tax Number that indicate Start.date (https://nypdonline.org/link/officer-profile)
- IDs that are attachead to a name, but the name is not found in the 2021 payroll all (but 1) are found to be last active before or during 2021. This indicated that the officer either resigned or was relieved of duty during the year 2021 , thus not having logged payroll information for the entire FY.
  - Only ID found in civil_summons_2021 that name was not found in 2021 payroll AND was last active AFTER 2021 was **CLAUDIA S FORREST (#965898)**
- Other IDs found in our civil_summons_2021, but were not associated with a name in the 2021 payroll, were laTst active before 2021. this raises a question of how they were able to give a summons in 2021 if they were relieved/left years before. 

LOGIC - 
from missing_id_list, if there is no initial in Middle.init, all any of the Officer.first.name and Officer.last.name 
of missing_id_list to 
First.name and Last.name of Payroll and connect the Final_Officer_Tax_ID associated from civil_summons

let me know how many merged and how many attached officer IDs

reattache these 

1. if missing_id_list Officer.first.name AND Officer.last.name == payroll_with_ids First.name AND Last.name,
ADD Final_Officer_Tax_ID from missing_id_list to Final_Officer_Tax_ID in payroll_with_ids 
    2. 2261 is the base rn in payroll_with_ids.Final_Officer_Tax_ID we want 2423
2. if missing_id_list 2



#### Completed Training : Consolidated column that contains a list of trainings that a PO completed from a selected group of training categories
**DELETE** 
#### Training Count : Total trainings completed by the PO from a selected group of training categories

**KEEP** 

ANALYSIS: Changed to training_count

In [ ]:
civil_summons_2021['Completed Training Names'].value_counts()

In [ ]:
# 308 officers have no training 
civil_summons_2021["Training Count"].value_counts().sort_index()

In [ ]:
# Change name 
civil_summons_2021 = civil_summons_2021.rename(
    columns={"Training Count": "training_count"}
)

In [ ]:
civil_summons_2021.training_count.head()

### After EDA, necessary indicators for analysis: 
- Keep necessary civil_summons_2021 variables for analysis (enforcement)
    - Ticket_Number 
    - Final_Officer_First_Name 
    - Final_Officer_Last_Name
    - Final_Officer_Tax_ID  

- Keep necessary OATH variables for analysis (enforcement)
    - O_Violation_Date
    - month_issued
    - O_Violation_Time
    - O_Violation_Location_Borough
    - O_Respondent_Address_Borough
    - O_Hearing_Status
    - O_Total_Violation_Amount
    - O_Payment_Status
    - O_Compliance_Status
    - O_Final_Charge

- Keep necessary disciplinary actions (DA/CC) variables (behavior) **Think about including later**
    - DA_charge
    - DA_disposition
    - DA_penalty

- Keep necessary arrest count (aa) variables (behavior)
    - aa_felony_arrest_count
    - aa_infraction_arrest_count
    - aa_misdemeanor_arrest_count
    - aa_other_arrest_count
    - aa_violation_arrest_count
    - total_arrest (still has 35 missing/not most updated) 

- Keep necessary award count (aa) variables (behavior)
    - commendation_award_count,
    - excellent_award_count
    - meritorious_award_count
    - meritorious_integrity_award_count
    - exceptional_merit_award_count
    - total_award_count

- Keep most updated rank, command, race, and gender (demographics)
    - roster_rank
    - roster_command
    - roster_race
    - roster_gender

- Keep training_count (behavior)

In [ ]:
civil_summons_2021.columns.to_list()

In [ ]:
civil_summons_2021.notna().sum()

In [ ]:
keep_cols = [

    # Officer identifiers
    "Ticket_Number",
    "Final_Officer_Tax_ID",
    "Officer.first.name",
    "Officer.last.name",
    'cs_first',
    'cs_last',
    'cs_middle',
    'pr_first',
    'pr_last',
    'pr_middle',
    # OATH enforcement variables
    'O_Violation_Date',
    'month_issued',
    'O_Violation_Time',
    'O_Issuing_Agency',
    'O_Respondent_First_Name',
    'O_Respondent_Last_Name',
    'O_Balance_Due',
    'O_Violation_Location_Borough',
    'O_Violation_Location_City',
    'O_Violation_Location_Zip_Code',
    'O_Violation_Location_State_Name',
    'O_Respondent_Address_Borough',
    'O_Hearing_Status',
    'O_Hearing_Result',
    'O_Scheduled_Hearing_Location',
    'O_Hearing_Date',
    'O_Hearing_Time',
    'O_Decision_Location_Borough',
    'O_Decision_Date',
    'O_Total_Violation_Amount',
    'O_Violation_Details',
    'O_Date_Judgment_Docketed',
    'O_Penalty_Imposed',
    'O_Paid_Amount',
    'O_Additional_Penalties_or_Late_Fees',
    'O_Compliance_Status',
   
    # Disciplinary actions (keep optional)
    "DA_charge",
    "DA_disposition",
    "DA_penalty",
    
    # Arrest counts
    "aa_felony_arrest_count",
    "aa_infraction_arrest_count",
    "aa_misdemeanor_arrest_count",
    "aa_other_arrest_count",
    "aa_violation_arrest_count",
    "aa_total_arrests",

    # Award counts
    "commendation_award_count",
    "excellent_award_count",
    "meritorious_award_count",
    "meritorious_integrity_award_count",
    "exceptional_merit_award_count",
    "total_award_count",
    
    # Roster demographic data
    "roster_rank",
    "roster_command",
    "roster_race",
    "roster_gender",
    'Total.CCRB.complaints',
    'Total.substantiated.CCRB.complaints',
    'Last.active',

    # Training
    "training_count",
    
   # Payroll
    'yrs_of_service', 
    'Reg.hours',
    'Overtime.hours'
   
]

civil_summons_2021 = civil_summons_2021[keep_cols].copy()


In [ ]:
civil_summons_2021.O_Respondent_Last_Name.info()

In [ ]:
civil_summons_2021 = civil_summons_2021.rename(
    columns={"Training Count": "training_count"}
)

In [ ]:

civil_summons_2021.columns.to_list()

## Civilian Complaint Review Board EDA
- **Source**: Civilian Complaint Review Board records, as available at NYC OpenData.This table is updated weekly, but the CCRB website will always have the most up-to-date information. Available CCRB reports are linked under “Reports” as they become available to Legal Aid Society. We currently have detailed reports for 6697 cases.
- **Description**: This table is updated weekly, but the CCRB website will always have the most up-to-date information. Available CCRB reports are linked under “Reports” as they become available to Legal Aid Society. We currently have detailed reports for 6697 cases. Reports are produced to Legal Aid Society as responses to FOIL requests by CAP, or obtained through other advocacy organizations like LatinoJustice and 50-a.org, and or when released by media outlets. Multiple versions of the same or similar reports may be available in some cases.
- **Scope**: 2021 - Present 
- **Key Variables:**
    - Complaint.category (FADO)
    - CCRB.complaint.disposition
    - Borough.of.incident
    - Victim.race.ethnicity.at.incident..legacy.
    - Victim.gender.at.incident
    - investigator_recommendation

**ANALYSIS: Out of our 2423 unique officers, 1252 officers have CCRB Allegations = 6255 allegations total** 

In [ ]:
# Create a list of the Tax IDs you want to filter by
tax_ids = civil_summons_2021["Final_Officer_Tax_ID"].dropna().unique().tolist()

ccrb_2021 = pd.read_csv(
    "data/ccrb_2021_present.tsv",
    sep="\t",
    dtype=str
)

ccrb_2021['Tax..'] = pd.to_numeric(
    ccrb_2021['Tax..'],
    errors='coerce'
)

# Filter rows where Tax_Registry_Number_Tax_ID matches
ccrb_filtered = ccrb_2021[ccrb_2021["Tax.."].isin(tax_ids)].copy()

In [ ]:
len(tax_ids)

In [ ]:
ccrb_2021

In [ ]:
len(ccrb_filtered)

In [ ]:
# Out of our 2423 unique officers, 1252 officers have CCRB Allegations = 6255 allegations total 
ccrb_filtered.describe()

In [ ]:
ccrb_filtered.columns.to_list()

In [ ]:
ccrb_filtered["Allegation.ID"].nunique()

In [ ]:
allegation_counts = ccrb_filtered['Tax..'].value_counts()
print(allegation_counts)

In [ ]:
# Key behavioral variables that we want from CCRB data connected to our officers
key_variables = [
    "Tax..",
    "Complaint.category",
    "CCRB.allegation.disposition",
    "Borough.of.incident",
    "Victim.race.ethnicity.at.incident..legacy.",
    "Victim.gender.at.incident",
    "investigator_recommendation"
]

# Keep only those columns
ccrb_filtered = ccrb_filtered[key_variables]

In [ ]:
ccrb_filtered.head(20)

#### Complaint.category: 
  - Abuse of Authority: Refers to abuse of police powers to intimidate or mistreat a civilian; for example, an officer’s refusal to provide name and badge number, or an improper “stop, question and frisk.” 
  - Discourtesy: Refers to cursing and using other foul language or gestures.
  - Force: Refers to the use of excessive or unnecessary force; behavior that includes punching or shoving and up to and including the use of deadly force. 
  - Offensive Language: Refers to slurs and derogatory remarks or gestures based upon race, ethnicity, religion, gender, sexual orientation, or physical disability.
  - Untruthful Statement
  
index:
  - 1
  - 2
  - 3
  - 4
  - 5

In [ ]:
ccrb_filtered['Complaint.category'].value_counts()

In [ ]:
ccrb_filtered['Complaint.category'].value_counts().plot(kind="bar", color="steelblue")

In [ ]:
ccrb_filtered['Complaint.category'].unique()

In [ ]:

ccrb_filtered['Complaint.category'] = (
    ccrb_filtered['Complaint.category']
    .astype(str)              # ensure string
    .str.strip()              # remove whitespace
    .astype(int)              # convert to integer
)

category_map = {
    1: "Abuse of Authority",
    2: "Discourtesy",
    3: "Force",
    4: "Offensive Language",
    5: "Untruthful Statement"
}

ccrb_filtered['Complaint.category'] = (
    ccrb_filtered['Complaint.category']
    .map(category_map)
)

ccrb_filtered['Complaint.category'].value_counts()


#### CCRB.allegation.disposition: Bin into 5 categories (Substantiated, Unable to Determine, Unfounded, Within NYPD Guidelines, Officers Unidentified): 

  - 1. Alleged Victim Unavailable 
  - 2. Alleged Victim Uncooperative
  - 3. Closed - Pending Litigation
  - 4. Complainant Unavailable
  - 5. Complainant Uncooperative
  - 6. Complaint Withdrawn
  - 7. Exonerated
  - 8. Miscellaneous
  - 30. Miscellaneous - Subject Deceased
  - 9. Miscellaneous - Subject Resigned
  - 10. Miscellaneous - Subject Retired
  - 11. Miscellaneous - Subject Terminated
  - 12. Officer(s) Unidentified
  - 13. OMB PEG Directive Closure
  - 29. SRAD Closure
  - 14. Substantiated (Charges)
  - 15. Substantiated (Command Discipline A)
  - 16. Substantiated (Command Discipline B)
  - 17. Substantiated (Command Discipline)
  - 18. Substantiated (Command Lvl Instructions)
  - 19. Substantiated (Formalized Training)
  - 20. Substantiated (Instructions)
  - 21. Substantiated (No Recommendations)
  - 22. Unable to Determine
  - 23. Unfounded
  - 24. Unsubstantiated
  - 25. Victim Unidentified
  - 26. Within NYPD Guidelines
  - 27. Witness Unavailable
  - 28. Witness Uncooperative
  


In [ ]:
disposition_bin = {
    # SUBSTANTIATED
    14: "SUBSTANTIATED",
    15: "SUBSTANTIATED",
    16: "SUBSTANTIATED",
    17: "SUBSTANTIATED",
    18: "SUBSTANTIATED",
    19: "SUBSTANTIATED",
    20: "SUBSTANTIATED",
    21: "SUBSTANTIATED",

    # also treat "mediated" as legit complaint
    9: "SUBSTANTIATED",   # if mediation attempted appears later
    32: "SUBSTANTIATED",  # if your dataset includes "Mediated"

    # UNABLE TO DETERMINE
    22: "UNABLE TO DETERMINE",

    # UNFOUNDED (and complainant not available)
    23: "UNFOUNDED",
    1: "UNFOUNDED",
    2: "UNFOUNDED",
    3: "UNFOUNDED",
    4: "UNFOUNDED",
    5: "UNFOUNDED",
    6: "UNFOUNDED",
    25: "UNFOUNDED",
    27: "UNFOUNDED",
    28: "UNFOUNDED",

    # WITHIN GUIDELINES (exonerated + administrative)
    26: "WITHIN GUIDELINES",
    7: "WITHIN GUIDELINES",
    8: "WITHIN GUIDELINES",
    9: "WITHIN GUIDELINES",
    10: "WITHIN GUIDELINES",
    11: "WITHIN GUIDELINES",
    13: "WITHIN GUIDELINES",
    29: "WITHIN GUIDELINES",
    30: "WITHIN GUIDELINES",

    # OFFICER UNIDENTIFIED
    12: "OFFICER UNIDENTIFIED"
}


ccrb_filtered["disposition_bin"] = (
    ccrb_filtered["CCRB.allegation.disposition"]
    .astype(int)
    .map(disposition_bin)
    .fillna("OTHER")
)

In [ ]:
ccrb_filtered["disposition_bin"].value_counts()

In [ ]:
plt.figure(figsize=(10,6))

ccrb_filtered["disposition_bin"].value_counts().plot(
    kind="bar",
    edgecolor="black",
    color="lightblue"                               
)

plt.title("Distribution of CCRB Disposition Bins", fontsize=16)
plt.xlabel("Disposition Category", fontsize=12)
plt.ylabel("Count of Allegations", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()


plt.savefig('EDA_figures/Distribution of CCRB Disposition Bins.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
ccrb_filtered['CCRB.allegation.disposition'].value_counts()

#### Borough.of.incident: shows where the complaints happened 
  - Bronx
  - Brooklyn
  - Manhattan
  - Outside NYC
  - Queens
  - Staten Island

index:
  - 1
  - 2
  - 3
  - 4
  - 5
  - 6

In [ ]:
ccrb_filtered['Borough.of.incident'].value_counts()

In [ ]:
ccrb_filtered['Borough.of.incident'].value_counts().plot(kind="bar", color="steelblue")

In [ ]:

borough_map = {
    "1": "Bronx",
    "2": "Brooklyn",
    "3": "Manhattan",
    "5": "Queens",
    "6": "Staten Island"
}


ccrb_filtered['Borough.of.incident'] = (
    ccrb_filtered['Borough.of.incident']
    .map(borough_map)
)

ccrb_filtered['Borough.of.incident'].value_counts()

In [ ]:
ccrb_filtered['Borough.of.incident'].value_counts()

#### Victim.race.ethnicity.at.incident..legacy.: 
  - American Indian
  - Asian
  - Black
  - Hispanic
  - Other Race
  - Refused
  - Unknown
  - White
  
  
  index:
  - 1
  - 2
  - 3
  - 4
  - 5
  - 6
  - 7
  - 8

In [ ]:
ccrb_filtered['Victim.race.ethnicity.at.incident..legacy.'].value_counts()

In [ ]:
# Plotting allegations victims by Race 
race_map = {
    1: "American Indian",
    2: "Asian",
    3: "Black",
    4: "Hispanic",
    5: "Other Race",
    6: "Refused",
    7: "Unknown",
    8: "White"
}

gender_map = {
    1: "Female", 
    2: "Male", 
    3: "TGNC"
}

# plot 
(
    ccrb_filtered["Victim.race.ethnicity.at.incident..legacy."] 
    .astype(float) 
    .map(race_map)
    .value_counts()
    .plot(kind="barh", title="Allegation Victims by Race Frequency ", figsize=(8,5))
)

plt.xlabel("Count")
plt.ylabel("Race/Ethnicity")

plt.savefig('EDA_figures/Allegation Victims by Race Frequency.png', dpi=300, bbox_inches='tight')
plt.show()

#### Victim.gender.at.incident: 
  - Female/Woman
  - Male/Man
  - TGNC / Other

index:
  - 1
  - 2
  - 3

In [ ]:
ccrb_filtered['Victim.gender.at.incident'].value_counts()

In [ ]:
ccrb_filtered['Victim.gender.at.incident'].value_counts().plot(kind="bar", color="steelblue")

In [ ]:
# Plotting victims by Race and Gender 
gender_map = {
    1: "Female", 
    2: "Male", 
    3: "TGNC" #transgender/non-conforming
}

# apply mappings
ccrb_filtered["Race"] = (
    ccrb_filtered["Victim.race.ethnicity.at.incident..legacy."]
    .astype(float)
    .map(race_map)
)

ccrb_filtered["Gender"] = (
    ccrb_filtered["Victim.gender.at.incident"]
    .astype(float)
    .map(gender_map)
)

# create a cross-tab (Race × Gender counts)
race_gender_counts = pd.crosstab(ccrb_filtered["Race"], ccrb_filtered["Gender"])

race_gender_counts.plot(
    kind="bar",
    figsize=(10,6),
    title="Allegation Victims by Race and Gender"
)

plt.xlabel("Race/Ethnicity")
plt.ylabel("Number of Allegations")
plt.xticks(rotation=45)
plt.legend(title="Gender")

plt.tight_layout()  # makes sure labels/titles aren’t cut off
plt.savefig('EDA_figures/Allegation Victims by Race and Gender.png', dpi=300, bbox_inches='tight')
plt.show()

#### Investigator Recommendation is the actual response to CCRB allegations 

In [ ]:
# Investigator Recommendation is the actuar response of the CCRB allegation 
ccrb_filtered["investigator_recommendation"].value_counts().sort_index().plot(kind="bar")

In [ ]:
ccrb_filtered["investigator_recommendation"].value_counts()

#### Merging CCRB behavior data with original civil_summons_2021
- Renaming ccrb_filtered tax.. to Final_Officer_Tax_ID
- Merge on Final_Officer_Tax_ID

In [ ]:
ccrb_filtered.info()

In [ ]:
ccrb_filtered.info()

In [ ]:
civil_summons_2021.head(15)

In [ ]:
# Lets get total_allegation 
allegation_counts

ccrb_filtered['Tax..'] = (
    pd.to_numeric(ccrb_filtered['Tax..'], errors='coerce')
    .astype('Int64')
)

civil_summons_2021['Final_Officer_Tax_ID'] = (
    pd.to_numeric(civil_summons_2021['Final_Officer_Tax_ID'], errors='coerce')
    .astype('Int64')
)

In [ ]:
allegation_counts

In [ ]:
civil_summons_2021['Final_Officer_Tax_ID']

In [ ]:
civil_summons_2021['CCRB_Total_Allegations'] = (
    civil_summons_2021['Final_Officer_Tax_ID']
    .map(allegation_counts)
    .fillna(0)
    .astype(int)
)

In [ ]:
civil_summons_2021.CCRB_Total_Allegations.value_counts()

In [ ]:
civil_summons_2021.columns.to_list()

In [ ]:
civil_summons_2021.to_csv("data/civil_summons_2021_v1.csv")

In [ ]:
# Sum Officer Total Summonses
civil_summons_2021['Officer_Total_Summonses'] = (
    civil_summons_2021
        .groupby('Final_Officer_Tax_ID')['Ticket_Number']
        .transform('count')
)

In [ ]:
civil_summons_2021.columns.to_list()

In [ ]:
ccrb_filtered.info()

#### ccrb['Victim.race.ethnicity.at.incident..legacy.'].value_counts()

1  - American Indian
2  - Asian
3  - Black
4  - Hispanic
5  - Other Race
6  - Refused
7  - Unknown
8  - White

In [ ]:
# 1. Create dummy variables for each victim race category
race_dummies = pd.get_dummies(ccrb_filtered['Race'], prefix='VictimRace')
race_dummies

In [ ]:
ccrb_with_dummies = pd.concat([ccrb_filtered[['Tax..']], race_dummies], axis=1)
    
# 3. Aggregate counts of each race by officer
ccrb_race_counts = ccrb_with_dummies.groupby('Tax..').sum().reset_index()

# 4. Rename columns to clearer names (optional)
ccrb_race_counts = ccrb_race_counts.rename(columns={
    'Tax..': 'Final_Officer_Tax_ID',
    'VictimRace_Black': 'CCRB_Black_Allegations',
    'VictimRace_White': 'CCRB_White_Allegations',
    'VictimRace_Asian': 'CCRB_Asian_Allegations',
    'VictimRace_Hispanic': 'CCRB_Hispanic_Allegations',
    'VictimRace_Other Race': 'CCRB_Other_Allegations',
    'VictimRace_Refused': 'CCRB_RefusedRace_Allegations',
    'VictimRace_Unknown': 'CCRB_Unknown_Allegations',
    'VictimRace_American Indian': 'CCRB_AmericanIndian_Allegations'
})


In [ ]:
civil_summons_2021['Final_Officer_Tax_ID'] = (
    civil_summons_2021['Final_Officer_Tax_ID']
    .astype(str)
    .str.strip()
)

ccrb_race_counts['Final_Officer_Tax_ID'] = (
    ccrb_race_counts['Final_Officer_Tax_ID']
    .astype(str)
    .str.strip()
)


In [ ]:
# Merge CCRB and civil summons on Final_Officer_Tax_ID 
civil_summons_2021 = civil_summons_2021.merge(
    ccrb_race_counts,
    on='Final_Officer_Tax_ID',
    how='left'
)

In [ ]:
# Save data 
civil_summons_2021.columns.to_list()

In [ ]:
civil_summons_2021.head()

In [ ]:
civil_summons_2021.CCRB_Black_Allegations.value_counts()

In [ ]:
# Map rank, race, and gender 
civil_summons_2021.roster_rank.info()

In [ ]:
civil_summons_2021.info()

In [ ]:
civil_summons_2021.select_dtypes('object').nunique()

In [ ]:
civil_summons_2021[['roster_rank','roster_race','roster_gender']].dtypes

In [ ]:
rank_map = {
    1: "CAPTAIN",
    2: "CHIEFS AND OTHER RANKS",
    3: "DEPUTY INSPECTOR",
    4: "DETECTIVE",
    5: "INACTIVE RANKS",
    6: "INSPECTOR",
    7: "LIEUTENANT",
    8: "POLICE OFFICER",
    9: "SERGEANT"
}

race_map = {
    1: "AMERICAN INDIAN",
    2: "ASIAN",
    3: "BLACK",
    4: "HISPANIC",
    5: "OTHER RACE",
    6: "WHITE"
}

gender_map = {
    1: "FEMALE",
    2: "MALE",
    3: "TGNC / OTHER"
}


In [ ]:
civil_summons_2021.roster_race.hist()

In [ ]:
civil_summons_2021.roster_rank.hist()

In [ ]:
# Convert floats to integers safely (NaN becomes NaN)
civil_summons_2021['roster_rank'] = civil_summons_2021['roster_rank'].astype('Int64')
civil_summons_2021['roster_race'] = civil_summons_2021['roster_race'].astype('Int64')
civil_summons_2021['roster_gender'] = civil_summons_2021['roster_gender'].astype('Int64')

# Map integer codes to text labels
civil_summons_2021['roster_rank'] = civil_summons_2021['roster_rank'].map(rank_map)
civil_summons_2021['roster_race'] = civil_summons_2021['roster_race'].map(race_map)
civil_summons_2021['roster_gender'] = civil_summons_2021['roster_gender'].map(gender_map)


In [ ]:
civil_summons_2021[['roster_rank','roster_race','roster_gender']].info()


In [ ]:
civil_summons_2021['roster_race'].head()

In [ ]:
# Replace missing values with "UNKNOWN"
civil_summons_2021['roster_rank']   = civil_summons_2021['roster_rank'].fillna("UNKNOWN")
civil_summons_2021['roster_race']   = civil_summons_2021['roster_race'].fillna("UNKNOWN")
civil_summons_2021['roster_gender'] = civil_summons_2021['roster_gender'].fillna("UNKNOWN")


In [ ]:
civil_summons_2021[['roster_rank','roster_race','roster_gender']].info()


In [ ]:
civil_summons_2021.select_dtypes('object').nunique()

In [ ]:
civil_summons_2021.columns.to_list()

In [ ]:
#civil_summons_2021.isna().sum()

In [ ]:
#(civil_summons_2021 == '.').sum()

In [ ]:
#final_charge = civil_summons_2021.O_Final_Charge.value_counts()

In [ ]:
#civil_summons_2021.info()

In [ ]:
civil_summons_2021['Final_Officer_Tax_ID'] = civil_summons_2021['Final_Officer_Tax_ID'].astype('Int64')
ccrb_filtered['Tax..'] = ccrb_filtered['Tax..'].astype('Int64')

In [ ]:
civil_summons_2021['Final_Officer_Tax_ID'].dtype, ccrb_filtered['Tax..'].dtype


In [ ]:
ccrb_filtered.info()

In [ ]:
# Mapping FADO 
# 1. Create dummy variables for each victim race category
fado_dummies = pd.get_dummies(ccrb_filtered['Complaint.category'], prefix='CCRB_fado')
fado_dummies

In [ ]:
# 2. Combine with Tax ID
ccrb_with_dummies = pd.concat([ccrb_filtered[['Tax..']], fado_dummies], axis=1)

# 3. Aggregate counts of each race by officer
ccrb_fado_counts = ccrb_with_dummies.groupby('Tax..').sum().reset_index()

# 4. Rename columns to clearer names (optional)
ccrb_fado_counts = ccrb_fado_counts.rename(columns={
    'Tax..': 'Final_Officer_Tax_ID'})


In [ ]:
civil_summons_2021 = civil_summons_2021.merge(
    ccrb_fado_counts,
    on='Final_Officer_Tax_ID',
    how='left'
)


In [ ]:
civil_summons_2021.columns.to_list()

In [ ]:
# Drop DA_charge, DA_disposition, DA_penalty
civil_summons_2021 = civil_summons_2021.drop(
    columns=[
        'DA_charge',
        'DA_disposition',
        'DA_penalty'
    ],
    errors='ignore'   
)


In [ ]:
# 1. Numeric columns → fill NaN with 0
num_cols = civil_summons_2021.select_dtypes(include=['number']).columns
civil_summons_2021[num_cols] = civil_summons_2021[num_cols].fillna(0)

# 2. Object columns → force missing to NaN
obj_cols = civil_summons_2021.select_dtypes(include=['object']).columns
civil_summons_2021[obj_cols] = civil_summons_2021[obj_cols].replace(
    ['.','', ' ', 'NA', 'NAN','N/A', 'nan', 'None'],
    np.nan
)


In [ ]:
civil_summons_2021.yrs_of_service.hist()

In [ ]:
civil_summons_2021.training_count.dtype

In [ ]:
# Adding back violation locations 
full_oath= pd.read_csv("data/full_2021_OATH.csv")

In [ ]:
full_oath.columns.to_list()

In [ ]:
oath_rename_map = {
    'Ticket Number': 'Ticket_Number',
    'Violation Location (Borough)': 'O_violation_location_borough',
    'Violation Location (Block No.)': 'O_violation_location_block_no',
    'Violation Location (Lot No.)': 'O_violation_location_lot_no',
    'Violation Location (House #)': 'O_violation_location_house_no',
    'Violation Location (Street Name)': 'O_violation_location_street_name',
    'Violation Location (Floor)': 'O_violation_location_floor',
    'Violation Location (City)': 'O_violation_location_city',
    'Violation Location (Zip Code)': 'O_violation_location_zip_code',
    'Violation Location (State Name)': 'O_violation_location_state',

    'Respondent Address (Borough)': 'O_respondent_address_borough',
}

In [ ]:
oath_cols = list(oath_rename_map.keys())

full_oath_subset = (
    full_oath[oath_cols]
    .rename(columns=oath_rename_map)
    .copy()
)


In [ ]:
for df in [civil_summons_2021, full_oath_subset]:
    df['Ticket_Number'] = pd.to_numeric(df['Ticket_Number'], errors='coerce')


In [ ]:
civil_summons_2021 = civil_summons_2021.merge(
    full_oath_subset,
    on='Ticket_Number',
    how='left'
)


In [ ]:
civil_summons_2021.columns.to_list()

In [ ]:
# 1. Identifiers & officer names
id_cols = [
    'Ticket_Number',
    'Final_Officer_Tax_ID',
    'Officer.first.name',
    'Officer.last.name',
    'cs_first', 'cs_middle', 'cs_last',
    'pr_first', 'pr_middle', 'pr_last',
    'Officer_Total_Summonses',
    'month_issued'
]

# 2. Roster / officer demographics & status
roster_cols = [
    'roster_rank',
    'roster_command',
    'roster_race',
    'roster_gender',
]

# 3. OATH / summons-related columns (ALL O_)
o_cols = [c for c in civil_summons_2021.columns if c.startswith('O_')]

# 4. Arrest activity
arrest_cols = [
    'aa_felony_arrest_count',
    'aa_misdemeanor_arrest_count',
    'aa_violation_arrest_count',
    'aa_infraction_arrest_count',
    'aa_other_arrest_count',
    'aa_total_arrests'
]

# 5. Awards
award_cols = [
    'commendation_award_count',
    'excellent_award_count',
    'meritorious_award_count',
    'meritorious_integrity_award_count',
    'exceptional_merit_award_count',
    'total_award_count'
]

# 6. CCRB
ccrb_cols = [
    'Total.CCRB.complaints',
    'Total.substantiated.CCRB.complaints',
    'CCRB_Total_Allegations',
    'CCRB_AmericanIndian_Allegations',
    'CCRB_Asian_Allegations',
    'CCRB_Black_Allegations',
    'CCRB_Hispanic_Allegations',
    'CCRB_White_Allegations',
    'CCRB_Other_Allegations',
    'CCRB_RefusedRace_Allegations',
    'CCRB_Unknown_Allegations',
    'CCRB_fado_Abuse of Authority',
    'CCRB_fado_Discourtesy',
    'CCRB_fado_Force',
    'CCRB_fado_Offensive Language',
    'CCRB_fado_Untruthful Statement'
]

# 7. Payroll / workload
payroll_cols = [
    'Reg.hours',
    'Overtime.hours',
    'Last.active',
    'yrs_of_service'
]


In [ ]:
ordered_cols = (
    id_cols +
    roster_cols +
    o_cols +
    arrest_cols +
    award_cols +
    ccrb_cols +
    payroll_cols
)

# Add any columns not explicitly listed
remaining_cols = [c for c in civil_summons_2021.columns if c not in ordered_cols]

final_cols = ordered_cols + remaining_cols


In [ ]:
civil_summons_2021 = civil_summons_2021[final_cols]

In [ ]:
civil_summons_2021.columns.tolist()

In [ ]:
civil_summons_2021.roster_race.value_counts()

In [ ]:
civil_summons_2021.roster_gender.value_counts()

In [ ]:
# How would you like to organize this? 
# Combine lieutenant, captain, inspector
# Police officer baseline
civil_summons_2021.roster_rank.value_counts()

In [ ]:
# Consolidate UNKNOWN into TGNC / OTHER
civil_summons_2021['roster_gender'] = civil_summons_2021['roster_gender'].replace(
    {'UNKNOWN': 'TGNC / OTHER'}
)
print(civil_summons_2021['roster_gender'].value_counts())

In [ ]:
# SAVE FINAL CIVIL_SUMMONS_2021_V1
civil_summons_2021.to_csv("data/civil_summons_2021_v4.csv")

In [ ]:
pd.set_option('display.max_rows', None)

missing_df = pd.DataFrame({
    'missing_count': civil_summons_2021.isnull().sum(),
    'missing_pct': civil_summons_2021.isnull().mean().mul(100).round(2)
}).sort_values('missing_pct', ascending=True)

missing_df 